# 03 — Model Training

**Goal:** Train an XGBoost classifier and understand its behaviour before formal evaluation.

### Why XGBoost?
- State-of-the-art on tabular classification benchmarks
- Native support for `scale_pos_weight` to handle class imbalance without resampling
- First-class compatibility with SHAP's `TreeExplainer` (exact, not approximate, SHAP values)
- Fast inference — critical for a sub-500ms prediction SLA

### Hyperparameter Rationale
| Parameter | Value | Reason |
|---|---|---|
| `n_estimators` | 300 | Sufficient depth without excessive training time |
| `max_depth` | 4 | Shallow trees generalise better on financial tabular data |
| `learning_rate` | 0.05 | Lower LR + more trees → better generalisation |
| `subsample` | 0.8 | Row subsampling reduces variance |
| `colsample_bytree` | 0.8 | Column subsampling adds regularisation |
| `min_child_weight` | 5 | Prevents splits on very small leaf nodes |
| `scale_pos_weight` | ~13.96 | Ratio of negative to positive samples |

In [ ]:
import sys
sys.path.insert(0, '../../..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

from ml.pipeline.preprocess import load_raw, build_preprocessor, FEATURE_COLUMNS, DISPLAY_NAMES

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

## 1. Load & Preprocess

In [ ]:
X, y = load_raw()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = build_preprocessor()
X_train_t = preprocessor.fit_transform(X_train)
X_test_t  = preprocessor.transform(X_test)

n_pos = int(y_train.sum())
n_neg = int((y_train == 0).sum())
scale_pos_weight = n_neg / n_pos
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

## 2. Stratified Cross-Validation

5-fold stratified CV on the training set to estimate generalisation before committing to the final model. Stratified folds preserve the 6.68% positive-class ratio across all folds.

In [ ]:
cv_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(cv_model, X_train_t, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)

print('5-Fold Stratified CV — ROC-AUC:')
for i, score in enumerate(cv_scores):
    print(f'  Fold {i+1}: {score:.4f}')
print(f'  Mean: {cv_scores.mean():.4f}  (+/- {cv_scores.std():.4f})')

## 3. Train Final Model

Train the final model on the full training set. We use the held-out test set as an eval set solely for monitoring — no early stopping, so there is no train/eval split contamination.

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

model.fit(
    X_train_t, y_train,
    eval_set=[(X_train_t, y_train), (X_test_t, y_test)],
    verbose=False,
)

results = model.evals_result()
train_auc = results['validation_0']['auc']
test_auc  = results['validation_1']['auc']

print(f'Final train AUC: {train_auc[-1]:.4f}')
print(f'Final test AUC:  {test_auc[-1]:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_auc, label='Train AUC', alpha=0.8, linewidth=1.5)
ax.plot(test_auc,  label='Test AUC',  alpha=0.8, linewidth=1.5)
ax.axhline(0.85, color='red', linestyle='--', alpha=0.6, label='Target (0.85)')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('ROC-AUC')
ax.set_title('Learning Curve — ROC-AUC by Boosting Round', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 4. XGBoost Feature Importance

XGBoost's built-in importance (gain) gives a rough ranking. Note that SHAP-based importance in notebook 04 is more reliable for correlated features.

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=[DISPLAY_NAMES[f] for f in FEATURE_COLUMNS]
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e74c3c' if importance[f] == importance.max() else '#3498db' for f in importance.index]
importance.plot(kind='barh', ax=ax, color=colors, alpha=0.85, edgecolor='none')
ax.set_title('XGBoost Feature Importance (Gain)', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Ranked features (highest gain first):')
for feat, val in importance.sort_values(ascending=False).items():
    print(f'  {feat:<35} {val:.4f}')

## Summary

- CV mean ROC-AUC indicates the model generalises well across folds
- The learning curve shows no significant overfitting (train/test AUC remain close)
- Credit Utilization dominates feature importance, consistent with domain knowledge
- Late payment features (30-59, 60-89, 90+ days) all rank highly

Proceed to `04_evaluation.ipynb` for full evaluation metrics, calibration, and SHAP analysis.